## 0. What this notebook is

Selected Phase-1 system, one notebook like the official starter.

The starter used `GPT2Config` + the GPT-2 tokenizer. This notebook does **not**. Tokenizer and models below are trained from scratch (vocab 8192, no `nn.Transformer`, no public checkpoint).

**Selected:** efficient (RoPE, GQA 16Q/4KV, SwiGLU), greedy + EOS.

`RUN_TRAINING = False` during the viva.


In [ ]:
import os, sys, types, random, json, math, zipfile
from pathlib import Path

_BEFORE = set(globals()) | {"sys", "types", "_BEFORE", "_register"}

def _register(name):
    global _BEFORE
    mod = types.ModuleType(name)
    mod.__name__ = name
    mod.__file__ = name + ".py"
    for key, value in list(globals().items()):
        if key not in _BEFORE:
            setattr(mod, key, value)
    sys.modules[name] = mod
    _BEFORE = set(globals()) | {"sys", "types", "_BEFORE", "_register"}
    print("in this notebook:", name)

print("This notebook is self-contained (starter style). RUN_TRAINING = False for a viva.")


## 1. Hugging Face login

Only if you re-download the CS-781 dataset.


In [ ]:
import os
try:
    from huggingface_hub import login
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF")
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)
    print("Hugging Face login successful.")
except Exception as exc:
    HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
    if HF_TOKEN:
        from huggingface_hub import login
        login(token=HF_TOKEN)
        print("Hugging Face login from environment.")
    else:
        print("No HF token in this session (fine for a viva walkthrough).")
        print(type(exc).__name__, str(exc)[:120])


## 2. Check GPU


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))


## 3. Configuration


In [ ]:
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = {
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
}
TOKEN_MIX = {"supreme_court": 0.50, "high_courts": 0.25, "tribunals": 0.15, "acts": 0.10}
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

BPE_VOCAB_SIZE = 8192
ARCH = "efficient"
BLOCK = 1024
N_EMBD = 1024
N_LAYER = 10
N_HEAD = 16
N_KV_HEAD = 4
TARGET_GENERATION_WORDS = 128
TEMPERATURE = 0.0
RUN_TRAINING = False
RUN_DATA = False
print("Architecture:", ARCH, "mix", TOKEN_MIX)


## 4. Tokenizer — Dice-weighted Legal BPE

Not GPT-2. Bytes 0–255 stay in the vocab. Word-end marker. `score(x,y) = 2 C(xy)^2 / (C(x)+C(y))`.


In [ ]:
"""Byte-level BPE for CS781 Capstone Phase 1.

This is trained from scratch on the capstone corpus. It does not use
SentencePiece, Hugging Face tokenizers, or the GPT-2 vocabulary.

Assignment 1 merges the pair that occurs most often. On legal text that
fills the vocabulary with pieces of very common words such as "the" and
"of", even when those letters usually belong to other words.

Here a pair must occur at least `min_pair_count` times. Among those pairs
the one with the highest Dice-weighted count is merged:

    score(x, y) = C(xy) * 2 * C(xy) / (C(x) + C(y))

The fraction is near 1 when x and y almost always appear as this pair, and
near 0 when they usually appear apart. Multiplying by C(xy) keeps frequent
pairs ahead of rare ones. If two pairs have the same score, the higher count
wins. If the counts are also equal, the lexicographically smaller pair wins,
which is the Assignment 1 tie-break.

Merges stay inside a whitespace-separated word. A word-end marker is part of
that word, so a frequent whole word can become one token. Spaces stay
spaces, and uppercase stays uppercase. The first four vocabulary ids are
<pad>, <unk>, <s>, </s>. Every byte 0-255 has an id, so ordinary text does
not need <unk>.
"""

from __future__ import annotations

import heapq
import json
import re
from collections import Counter


END = 256  # word-end marker, not a UTF-8 byte
PAD_ID = 0
UNK_ID = 1
BOS_ID = 2
EOS_ID = 3
SPECIALS = ("<pad>", "<unk>", "<s>", "</s>")
# Model ids 0-3 are the specials above. Internal symbol 0 (byte 0) is model id 4.
ID_OFFSET = 4
BASE_SYMBOLS = 257  # bytes 0-255 plus END
TOKEN_RE = re.compile(r"\s+|\S+")


def iter_texts(records):
    """Yield strings from a list, a dict of records, or a single record."""
    if isinstance(records, list):
        rows = records
    elif isinstance(records, dict):
        if isinstance(records.get("text"), str):
            rows = [records]
        else:
            rows = list(records.values())
    else:
        raise TypeError("records must be a list or a dict")
    for row in rows:
        if isinstance(row, str):
            yield row
        elif isinstance(row, dict) and isinstance(row.get("text"), str):
            yield row["text"]


class LegalBPE:
    def __init__(self):
        self.score_name = "dice"
        self.min_pair_count = 2
        self.requested_vocab_size = ID_OFFSET + BASE_SYMBOLS
        self.merges = []  # list of (left_internal, right_internal), in learn order
        self.symbol_bytes = {i: (i,) for i in range(256)}
        self.symbol_bytes[END] = (END,)
        self.merge_rank = {}
        self.merge_to = {}
        self.keys = {}
        self.history = []
        self._cache = {}
        self._rebuild_tables()

    # ------------------------------------------------------------------
    # Public attributes the Phase 1 notebook uses
    # ------------------------------------------------------------------

    @property
    def pad_token_id(self):
        return PAD_ID

    @property
    def unk_token_id(self):
        return UNK_ID

    @property
    def bos_token_id(self):
        return BOS_ID

    @property
    def eos_token_id(self):
        return EOS_ID

    pad_token = SPECIALS[0]
    unk_token = SPECIALS[1]
    bos_token = SPECIALS[2]
    eos_token = SPECIALS[3]

    def __len__(self):
        return ID_OFFSET + BASE_SYMBOLS + len(self.merges)

    # ------------------------------------------------------------------
    # Training
    # ------------------------------------------------------------------

    @classmethod
    def train(cls, texts, vocab_size=8192, min_pair_count=2, score="dice", max_chars=None):
        if score not in ("dice", "frequency"):
            raise ValueError("score must be 'dice' or 'frequency'")
        minimum = ID_OFFSET + BASE_SYMBOLS + 1
        if vocab_size < minimum:
            raise ValueError(f"vocab_size must be at least {minimum}")

        model = cls()
        model.score_name = score
        model.min_pair_count = int(min_pair_count)
        model.requested_vocab_size = int(vocab_size)
        target_merges = int(vocab_size) - (ID_OFFSET + BASE_SYMBOLS)

        word_freq = Counter()
        documents = 0
        seen_chars = 0
        for text in texts:
            documents += 1
            if not isinstance(text, str) or text == "":
                continue
            if max_chars is not None and seen_chars >= max_chars:
                break
            seen_chars += len(text)
            for tok in TOKEN_RE.findall(text):
                word_freq[model._token_symbols(tok)] += 1
            if documents % 100000 == 0:
                print(f"BPE counted {documents:,} documents, {len(word_freq):,} unique words")

        if not word_freq:
            raise ValueError("BPE training saw no text")

        print(
            f"BPE documents: {documents:,} | characters: {seen_chars:,} | "
            f"unique words: {len(word_freq):,} | score: {score} | target vocab: {vocab_size}"
        )
        model._learn(word_freq, target_merges)
        print(f"BPE finished. Vocabulary size: {len(model):,}. Merges: {len(model.merges):,}.")
        for left, right, count, value in model.history[:12]:
            print(f"  merge {left!r} + {right!r}  count={count}  score={value:.2f}")
        return model

    def _learn(self, word_freq, target_merges):
        self.word_freq = dict(word_freq)
        self.pair_counts = Counter()
        self.pair_words = {}
        self.sym_counts = Counter()
        self.sym_pairs = {}
        self.version = {}
        self.heap = []
        self.history = []
        self.merges = []
        self.merge_rank = {}
        self.merge_to = {}
        self._cache = {}
        self.symbol_bytes = {i: (i,) for i in range(256)}
        self.symbol_bytes[END] = (END,)
        self.keys = {i: self._key_for(i) for i in range(257)}

        dirty_pairs = set()
        dirty_syms = set()
        for word, freq in self.word_freq.items():
            self._register(word, freq, dirty_pairs, dirty_syms)
        self._rebuild_heap()

        next_id = 257
        while len(self.merges) < target_merges:
            if len(self.heap) > max(1000, 8 * len(self.pair_counts)):
                self._rebuild_heap()
            chosen = self._pop_best()
            if chosen is None:
                print(
                    f"BPE stopped early at vocab {len(self)}. "
                    f"No pair occurs at least {self.min_pair_count} times."
                )
                break
            left, right = chosen
            count = self.pair_counts[chosen]
            value = self._score(chosen)
            new_id = next_id
            next_id += 1
            self.symbol_bytes[new_id] = self.symbol_bytes[left] + self.symbol_bytes[right]
            self.keys[new_id] = self._key_for(new_id)
            self.merges.append((left, right))
            self.history.append((self._pretty(left), self._pretty(right), int(count), float(value)))

            dirty_pairs = set()
            dirty_syms = set()
            for word in list(self.pair_words.get(chosen, ())):
                freq = self.word_freq.pop(word, None)
                if freq is None:
                    continue
                self._unregister(word, freq, dirty_pairs, dirty_syms)
                merged = _merge_symbols(word, left, right, new_id)
                if merged in self.word_freq:
                    self.word_freq[merged] += freq
                    self._register(merged, freq, dirty_pairs, dirty_syms)
                else:
                    self.word_freq[merged] = freq
                    self._register(merged, freq, dirty_pairs, dirty_syms)

            affected = set(dirty_pairs)
            for sym in dirty_syms:
                affected.update(self.sym_pairs.get(sym, ()))
            affected.add(chosen)
            for pair in affected:
                self._push(pair)

            done = len(self.merges)
            if done <= 10 or done % 500 == 0:
                print(
                    f"  [{done}/{target_merges}] {self.history[-1][0]!r} + "
                    f"{self.history[-1][1]!r}  count={count}  score={value:.2f}"
                )

        self._rebuild_tables()
        # Training indexes are large. Drop them before save.
        self.word_freq = {}
        self.pair_counts = Counter()
        self.pair_words = {}
        self.sym_counts = Counter()
        self.sym_pairs = {}
        self.heap = []

    def _register(self, word, freq, dirty_pairs, dirty_syms):
        for sym in word:
            self.sym_counts[sym] += freq
            dirty_syms.add(sym)
        for pair, times in Counter(zip(word, word[1:])).items():
            self.pair_counts[pair] += freq * times
            self.pair_words.setdefault(pair, set()).add(word)
            for sym in pair:
                self.sym_pairs.setdefault(sym, set()).add(pair)
            dirty_pairs.add(pair)

    def _unregister(self, word, freq, dirty_pairs, dirty_syms):
        for sym in word:
            self.sym_counts[sym] -= freq
            if self.sym_counts[sym] <= 0:
                del self.sym_counts[sym]
            dirty_syms.add(sym)
        for pair, times in Counter(zip(word, word[1:])).items():
            self.pair_counts[pair] -= freq * times
            words = self.pair_words.get(pair)
            if words is not None:
                words.discard(word)
                if not words:
                    del self.pair_words[pair]
            if self.pair_counts[pair] <= 0:
                del self.pair_counts[pair]
                for sym in pair:
                    bucket = self.sym_pairs.get(sym)
                    if bucket is not None:
                        bucket.discard(pair)
                        if not bucket:
                            del self.sym_pairs[sym]
                self.version[pair] = self.version.get(pair, 0) + 1
            else:
                dirty_pairs.add(pair)

    def _score(self, pair):
        count = self.pair_counts.get(pair, 0)
        if count < self.min_pair_count:
            return None
        if self.score_name == "frequency":
            return float(count)
        left = self.sym_counts.get(pair[0], 0)
        right = self.sym_counts.get(pair[1], 0)
        if left <= 0 or right <= 0:
            return None
        return (2.0 * count * count) / (left + right)

    def _push(self, pair):
        value = self._score(pair)
        if value is None:
            return
        version = self.version.get(pair, 0) + 1
        self.version[pair] = version
        count = self.pair_counts[pair]
        heapq.heappush(
            self.heap,
            (
                -value,
                -count,
                self.keys[pair[0]],
                self.keys[pair[1]],
                pair[0],
                pair[1],
                version,
            ),
        )

    def _rebuild_heap(self):
        self.heap = []
        for pair in list(self.pair_counts):
            self._push(pair)

    def _pop_best(self):
        while self.heap:
            _neg_value, _neg_count, _left_key, _right_key, left, right, version = heapq.heappop(self.heap)
            pair = (left, right)
            if self.version.get(pair) != version:
                continue
            if self._score(pair) is None:
                continue
            return pair
        return None

    def _rebuild_tables(self):
        self.merge_rank = {}
        self.merge_to = {}
        self.symbol_bytes = {i: (i,) for i in range(256)}
        self.symbol_bytes[END] = (END,)
        next_id = 257
        for rank, (left, right) in enumerate(self.merges):
            self.merge_rank[(left, right)] = rank
            self.merge_to[(left, right)] = next_id
            self.symbol_bytes[next_id] = self.symbol_bytes[left] + self.symbol_bytes[right]
            next_id += 1
        self.keys = {i: self._key_for(i) for i in self.symbol_bytes}
        self._cache = {}

    def _key_for(self, sym):
        parts = []
        for piece in self.symbol_bytes[sym]:
            parts.append("</w>" if piece == END else f"{piece:02x}")
        return "".join(parts)

    def _pretty(self, sym):
        raw = []
        end = False
        for piece in self.symbol_bytes[sym]:
            if piece == END:
                end = True
            else:
                raw.append(piece)
        try:
            text = bytes(raw).decode("utf-8")
            shown = text.encode("ascii", errors="backslashreplace").decode("ascii")
        except UnicodeDecodeError:
            shown = "".join(f"<{byte:02x}>" for byte in raw)
        return shown + "</w>" if end else shown

    def _token_symbols(self, token):
        raw = tuple(token.encode("utf-8"))
        if token.isspace():
            return raw
        return raw + (END,)

    # ------------------------------------------------------------------
    # Encode / decode
    # ------------------------------------------------------------------

    def encode(self, text, add_special_tokens=False):
        ids = []
        if add_special_tokens:
            ids.append(BOS_ID)
        if text:
            for tok in TOKEN_RE.findall(text):
                ids.extend(self._encode_token(tok))
        if add_special_tokens:
            ids.append(EOS_ID)
        return ids

    def encode_with_offsets(self, text):
        """Map text to token ids and the exact character span of each token.

        Merges never cross a whitespace boundary, and the word-end marker
        does not consume a character. A span is a half-open [start, end)
        into `text`, which is the same convention as the Phase 2 gold files.
        """
        ids = []
        spans = []
        for match in TOKEN_RE.finditer(text):
            token = match.group()
            origin = match.start()
            char_of_byte = []
            for index, char in enumerate(token):
                char_of_byte.extend([origin + index] * len(char.encode("utf-8")))
            byte_at = 0
            for model_id in self._encode_token(token):
                pieces = self.symbol_bytes.get(int(model_id) - ID_OFFSET, ())
                nbytes = sum(1 for piece in pieces if piece != END)
                if nbytes <= 0 or byte_at >= len(char_of_byte):
                    continue
                stop = min(byte_at + nbytes, len(char_of_byte))
                left = char_of_byte[byte_at]
                right = char_of_byte[stop - 1] + 1
                byte_at = stop
                if right > left:
                    ids.append(int(model_id))
                    spans.append((left, right))
        return ids, spans

    def _encode_token(self, token):
        cached = self._cache.get(token)
        if cached is None:
            symbols = list(self._token_symbols(token))
            while len(symbols) >= 2:
                best_rank = None
                best_index = None
                for i in range(len(symbols) - 1):
                    rank = self.merge_rank.get((symbols[i], symbols[i + 1]))
                    if rank is not None and (best_rank is None or rank < best_rank):
                        best_rank = rank
                        best_index = i
                if best_index is None:
                    break
                pair = (symbols[best_index], symbols[best_index + 1])
                symbols = symbols[:best_index] + [self.merge_to[pair]] + symbols[best_index + 2 :]
            cached = tuple(sym + ID_OFFSET for sym in symbols)
            self._cache[token] = cached
        return cached

    def decode(self, ids, skip_special_tokens=True):
        raw = []
        for model_id in ids:
            model_id = int(model_id)
            if model_id < ID_OFFSET:
                continue
            internal = model_id - ID_OFFSET
            pieces = self.symbol_bytes.get(internal)
            if pieces is None:
                continue
            for piece in pieces:
                if piece != END:
                    raw.append(piece)
        return bytes(raw).decode("utf-8", errors="replace")

    def batch_encode(self, texts, max_length=None, padding="left"):
        """Pad a batch for generation. Keep the end of a long prompt."""
        rows = []
        for text in texts:
            ids = self.encode(text, add_special_tokens=False)
            if max_length is not None and len(ids) > max_length:
                ids = ids[-max_length:]
            rows.append(ids)
        width = max((len(row) for row in rows), default=0)
        input_ids = []
        attention_mask = []
        for row in rows:
            pad = width - len(row)
            if padding == "left":
                input_ids.append([PAD_ID] * pad + row)
                attention_mask.append([0] * pad + [1] * len(row))
            else:
                input_ids.append(row + [PAD_ID] * pad)
                attention_mask.append([1] * len(row) + [0] * pad)
        return input_ids, attention_mask

    # ------------------------------------------------------------------
    # Save / load
    # ------------------------------------------------------------------

    def save(self, path):
        payload = {
            "format": "cs781-legal-bpe-v1",
            "score": self.score_name,
            "min_pair_count": self.min_pair_count,
            "vocab_size": len(self),
            "specials": list(SPECIALS),
            "merges": self.merges,
            "preview": [f"{left} + {right}" for left, right, _count, _value in self.history[:40]],
        }
        with open(path, "w", encoding="utf-8") as handle:
            json.dump(payload, handle, ensure_ascii=False)
        print("Saved BPE to", path)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as handle:
            payload = json.load(handle)
        if payload.get("format") != "cs781-legal-bpe-v1":
            raise ValueError(f"{path} is not a cs781-legal-bpe-v1 file")
        model = cls()
        model.score_name = payload.get("score", "dice")
        model.min_pair_count = int(payload.get("min_pair_count", 2))
        model.merges = [(int(left), int(right)) for left, right in payload["merges"]]
        model.requested_vocab_size = int(payload.get("vocab_size", len(model)))
        model.history = []
        model._rebuild_tables()
        return model


def _merge_symbols(word, left, right, new_id):
    merged = []
    index = 0
    size = len(word)
    while index < size:
        if index + 1 < size and word[index] == left and word[index + 1] == right:
            merged.append(new_id)
            index += 2
        else:
            merged.append(word[index])
            index += 1
    return tuple(merged)


def _check():
    sticky = ["ab"] * 6
    loose = ["mn", "mp", "mq", "mr", "ms"] * 10
    one_merge = ID_OFFSET + BASE_SYMBOLS + 1

    dice = LegalBPE.train(sticky + loose, vocab_size=one_merge, min_pair_count=2, score="dice")
    freq = LegalBPE.train(sticky + loose, vocab_size=one_merge, min_pair_count=2, score="frequency")
    assert dice.history[0][0] == "a" and dice.history[0][1] == "b", dice.history[0]
    assert freq.history[0][0] in list("mnmpqrms") or freq.history[0][0].startswith("m"), freq.history[0]
    assert not (freq.history[0][0] == "a" and freq.history[0][1] == "b"), freq.history[0]

    samples = [
        "The Supreme Court held that the petitioner is entitled to relief.",
        "Section 302 of the Indian Penal Code\nwas applied.",
        "court Court COURT",
        "न्यायालय ने आदेश दिया।",
        "  spaced   words\tand tabs",
    ]
    full = LegalBPE.train(samples * 30, vocab_size=400, min_pair_count=2, score="dice")
    for text in samples:
        assert full.decode(full.encode(text)) == text, text
    assert full.encode("Court") != full.encode("court")
    joined = full.decode(full.encode("ab cd"))
    assert joined == "ab cd"
    # A space is its own token, so a merge cannot glue the b in "ab" to the c in "cd".
    encoded = full.encode("ab cd")
    assert len(encoded) >= 2

    path = "_legal_bpe_selfcheck.json"
    full.save(path)
    loaded = LegalBPE.load(path)
    for text in samples:
        assert loaded.decode(loaded.encode(text)) == text
    assert len(loaded) == len(full)
    ids, mask = loaded.batch_encode(["short", "a longer prompt"], max_length=8, padding="left")
    assert len(ids[0]) == len(ids[1])
    assert mask[0][0] == 0 or len(loaded.encode("short")) == len(ids[0])
    import os
    os.remove(path)
    print("legal_bpe self-check passed")


In [ ]:
_register('legal_bpe')


## 5. Encode a legal string


In [ ]:
samples = [
    "The Supreme Court held that the petitioner",
    "AIR 1995 SC 123",
    "Section 302 of the Indian Penal Code",
]
print("specials", SPECIALS)
print("word-end marker id", END)


## 6. Four from-scratch LMs

Selected zip uses **efficient**. `build_model` refuses a count outside 50M–300M.


In [ ]:
"""Four Phase-1 language models, all trained from scratch.

They share a token embedding with the output layer. None of them loads a
pretrained checkpoint, and none of them uses nn.Transformer or
nn.MultiheadAttention.

The four designs are different in structure, not just in width or learning rate:

decoder
    Causal Transformer. A token may look only at earlier tokens. Position is a
    learned vector. The feed-forward layer is a GELU MLP. Pre-norm LayerNorm.

encdec
    The first half of a training block is read by a bidirectional encoder.
    A causal decoder writes the second half and cross-attends to the encoder.
    At test time the prompt is encoded, then the continuation is decoded.

hybrid
    Same size as the decoder, but most layers see only a local window of
    earlier tokens. Every fourth layer is full causal attention, so a long
    legal reference can still move through the network.

efficient
    RoPE instead of learned positions, RMSNorm instead of LayerNorm, SwiGLU
    instead of GELU, and grouped-query attention: 16 query heads share 4
    key/value heads.
"""

from __future__ import annotations

import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint


VOCAB_SIZE = 8192
BLOCK = 1024


def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters())


class RMSNorm(nn.Module):
    def __init__(self, n_embd, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(n_embd))
        self.eps = eps

    def forward(self, x):
        scale = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return x * scale * self.weight


def _init_linear(module):
    if isinstance(module, nn.Linear):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)


def apply_rope(x):
    # x: batch, heads, time, dim. Pairs (0,1), (2,3), ... are rotated by position.
    _batch, _heads, steps, dim = x.shape
    half = dim // 2
    freq = 1.0 / (10000 ** (torch.arange(half, device=x.device, dtype=torch.float32) / half))
    pos = torch.arange(steps, device=x.device, dtype=torch.float32)
    angles = torch.outer(pos, freq)
    cos = angles.cos()[None, None, :, :]
    sin = angles.sin()[None, None, :, :]
    even = x[..., 0::2].float()
    odd = x[..., 1::2].float()
    rotated_even = even * cos - odd * sin
    rotated_odd = even * sin + odd * cos
    stacked = torch.stack((rotated_even, rotated_odd), dim=-1)
    return stacked.flatten(-2).to(dtype=x.dtype)


class CausalAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, window=None):
        super().__init__()
        if n_embd % n_head != 0:
            raise ValueError("n_embd must divide into equal heads")
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.window = window
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.bidirectional = False
        causal = torch.triu(torch.ones(block_size, block_size, dtype=torch.bool), diagonal=1)
        self.register_buffer("causal", causal, persistent=False)

    def forward(self, x, pad_mask=None):
        batch, steps, channels = x.shape
        qkv = self.qkv(x).view(batch, steps, 3, self.n_head, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        query, key, value = qkv[0], qkv[1], qkv[2]
        scores = (query @ key.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if not self.bidirectional:
            banned = self.causal[:steps, :steps]
            if self.window is not None:
                positions = torch.arange(steps, device=x.device)
                too_far = positions[None, :] < (positions[:, None] - (self.window - 1))
                banned = banned | too_far
            scores = scores.masked_fill(banned, float("-inf"))
        if pad_mask is not None:
            scores = scores.masked_fill(~pad_mask[:, None, None, :], float("-inf"))
        weights = torch.nan_to_num(torch.softmax(scores, dim=-1), nan=0.0)
        mixed = (weights @ value).transpose(1, 2).contiguous().view(batch, steps, channels)
        return self.proj(mixed)


class GroupedQueryAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, block_size):
        super().__init__()
        if n_embd % n_head != 0 or n_head % n_kv_head != 0:
            raise ValueError("heads must divide the width and the kv heads")
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.head_dim = n_embd // n_head
        self.q = nn.Linear(n_embd, n_embd, bias=False)
        self.k = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.v = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.bidirectional = False
        causal = torch.triu(torch.ones(block_size, block_size, dtype=torch.bool), diagonal=1)
        self.register_buffer("causal", causal, persistent=False)

    def forward(self, x, pad_mask=None):
        batch, steps, _channels = x.shape
        query = self.q(x).view(batch, steps, self.n_head, self.head_dim).transpose(1, 2)
        key = self.k(x).view(batch, steps, self.n_kv_head, self.head_dim).transpose(1, 2)
        value = self.v(x).view(batch, steps, self.n_kv_head, self.head_dim).transpose(1, 2)
        query = apply_rope(query)
        key = apply_rope(key)
        groups = self.n_head // self.n_kv_head
        key = key.repeat_interleave(groups, dim=1)
        value = value.repeat_interleave(groups, dim=1)
        scores = (query @ key.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if not self.bidirectional:
            scores = scores.masked_fill(self.causal[:steps, :steps], float("-inf"))
        if pad_mask is not None:
            scores = scores.masked_fill(~pad_mask[:, None, None, :], float("-inf"))
        weights = torch.nan_to_num(torch.softmax(scores, dim=-1), nan=0.0)
        mixed = (weights @ value).transpose(1, 2).contiguous().view(batch, steps, -1)
        return self.proj(mixed)


class CrossAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.q = nn.Linear(n_embd, n_embd, bias=False)
        self.k = nn.Linear(n_embd, n_embd, bias=False)
        self.v = nn.Linear(n_embd, n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)

    def forward(self, x, memory, memory_mask=None):
        batch, steps, channels = x.shape
        mem_steps = memory.size(1)
        query = self.q(x).view(batch, steps, self.n_head, self.head_dim).transpose(1, 2)
        key = self.k(memory).view(batch, mem_steps, self.n_head, self.head_dim).transpose(1, 2)
        value = self.v(memory).view(batch, mem_steps, self.n_head, self.head_dim).transpose(1, 2)
        scores = (query @ key.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if memory_mask is not None:
            scores = scores.masked_fill(~memory_mask[:, None, None, :], float("-inf"))
        weights = torch.nan_to_num(torch.softmax(scores, dim=-1), nan=0.0)
        mixed = (weights @ value).transpose(1, 2).contiguous().view(batch, steps, channels)
        return self.proj(mixed)


class BidirectionalAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)

    def forward(self, x, pad_mask=None):
        batch, steps, channels = x.shape
        qkv = self.qkv(x).view(batch, steps, 3, self.n_head, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        query, key, value = qkv[0], qkv[1], qkv[2]
        scores = (query @ key.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if pad_mask is not None:
            scores = scores.masked_fill(~pad_mask[:, None, None, :], float("-inf"))
        weights = torch.nan_to_num(torch.softmax(scores, dim=-1), nan=0.0)
        mixed = (weights @ value).transpose(1, 2).contiguous().view(batch, steps, channels)
        return self.proj(mixed)


class GeluMLP(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.up = nn.Linear(n_embd, 4 * n_embd)
        self.down = nn.Linear(4 * n_embd, n_embd)

    def forward(self, x):
        return self.down(F.gelu(self.up(x)))


class SwiGLU(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        hidden = int(8 * n_embd / 3)
        self.gate = nn.Linear(n_embd, hidden, bias=False)
        self.up = nn.Linear(n_embd, hidden, bias=False)
        self.down = nn.Linear(hidden, n_embd, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class DecoderBlock(nn.Module):
    def __init__(self, n_embd, n_head, block_size, window=None, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(n_embd)
        self.attn = CausalAttention(n_embd, n_head, block_size, window=window)
        self.norm2 = nn.LayerNorm(n_embd)
        self.mlp = GeluMLP(n_embd)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, pad_mask=None):
        x = x + self.drop(self.attn(self.norm1(x), pad_mask=pad_mask))
        x = x + self.drop(self.mlp(self.norm2(x)))
        return x


class EfficientBlock(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, block_size, dropout=0.1):
        super().__init__()
        self.norm1 = RMSNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, block_size)
        self.norm2 = RMSNorm(n_embd)
        self.mlp = SwiGLU(n_embd)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, pad_mask=None):
        x = x + self.drop(self.attn(self.norm1(x), pad_mask=pad_mask))
        x = x + self.drop(self.mlp(self.norm2(x)))
        return x


class EncoderBlock(nn.Module):
    def __init__(self, n_embd, n_head, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(n_embd)
        self.attn = BidirectionalAttention(n_embd, n_head)
        self.norm2 = nn.LayerNorm(n_embd)
        self.mlp = GeluMLP(n_embd)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, pad_mask=None):
        x = x + self.drop(self.attn(self.norm1(x), pad_mask=pad_mask))
        x = x + self.drop(self.mlp(self.norm2(x)))
        return x


class DecoderCrossBlock(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(n_embd)
        self.self_attn = CausalAttention(n_embd, n_head, block_size)
        self.norm2 = nn.LayerNorm(n_embd)
        self.cross_attn = CrossAttention(n_embd, n_head)
        self.norm3 = nn.LayerNorm(n_embd)
        self.mlp = GeluMLP(n_embd)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, memory, pad_mask=None, memory_mask=None):
        x = x + self.drop(self.self_attn(self.norm1(x), pad_mask=pad_mask))
        x = x + self.drop(self.cross_attn(self.norm2(x), memory, memory_mask=memory_mask))
        x = x + self.drop(self.mlp(self.norm3(x)))
        return x


class TiedLM(nn.Module):
    def __init__(self, vocab_size, n_embd, block_size, use_positions=True):
        super().__init__()
        self.block_size = block_size
        self.embed = nn.Embedding(vocab_size, n_embd)
        self.pos = nn.Embedding(block_size, n_embd) if use_positions else None
        self.drop = nn.Dropout(0.1)
        self.norm = None

    def token_plus_position(self, idx):
        x = self.embed(idx)
        if self.pos is not None:
            steps = idx.size(1)
            positions = torch.arange(steps, device=idx.device)
            x = x + self.pos(positions)
        return self.drop(x)

    def logits_from(self, x):
        x = self.norm(x)
        return F.linear(x, self.embed.weight)

    def run_blocks(self, blocks, x, pad_mask=None):
        for block in blocks:
            if self.training:
                x = checkpoint(block, x, pad_mask, use_reentrant=False)
            else:
                x = block(x, pad_mask=pad_mask)
        return x


class DecoderLM(TiedLM):
    def __init__(self, vocab_size=VOCAB_SIZE, n_embd=1024, n_layer=8, n_head=16, block_size=BLOCK, windowed=False):
        super().__init__(vocab_size, n_embd, block_size, use_positions=True)
        blocks = []
        for layer in range(n_layer):
            window = None
            if windowed and layer % 4 != 3:
                window = 256
            blocks.append(DecoderBlock(n_embd, n_head, block_size, window=window))
        self.blocks = nn.ModuleList(blocks)
        self.norm = nn.LayerNorm(n_embd)
        self.apply(_init_linear)

    def forward(self, idx, labels=None, pad_mask=None):
        if idx.size(1) > self.block_size:
            raise ValueError("sequence is longer than the context")
        x = self.run_blocks(self.blocks, self.token_plus_position(idx), pad_mask)
        logits = self.logits_from(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)), labels[:, 1:].reshape(-1))
        return logits, loss


class EncoderDecoderLM(TiedLM):
    def __init__(self, vocab_size=VOCAB_SIZE, n_embd=768, n_layer=6, n_head=12, block_size=BLOCK):
        super().__init__(vocab_size, n_embd, block_size, use_positions=True)
        self.encoder = nn.ModuleList(EncoderBlock(n_embd, n_head) for _ in range(n_layer))
        self.decoder = nn.ModuleList(DecoderCrossBlock(n_embd, n_head, block_size) for _ in range(n_layer))
        self.norm = nn.LayerNorm(n_embd)
        self.enc_norm = nn.LayerNorm(n_embd)
        self.apply(_init_linear)

    def encode(self, idx, pad_mask=None):
        x = self.token_plus_position(idx)
        x = self.run_blocks(self.encoder, x, pad_mask)
        return self.enc_norm(x)

    def decode(self, idx, memory, pad_mask=None, memory_mask=None):
        x = self.token_plus_position(idx)
        for block in self.decoder:
            if self.training:
                x = checkpoint(block, x, memory, pad_mask, memory_mask, use_reentrant=False)
            else:
                x = block(x, memory, pad_mask=pad_mask, memory_mask=memory_mask)
        return self.logits_from(x)

    def forward(self, idx, labels=None, pad_mask=None):
        steps = idx.size(1)
        if steps < 2 or steps > self.block_size:
            raise ValueError("encoder-decoder training needs a block of length 2..context")
        split = steps // 2
        memory = self.encode(idx[:, :split], None if pad_mask is None else pad_mask[:, :split])
        dec_in = idx[:, split - 1 : -1]
        target = idx[:, split:]
        logits = self.decode(dec_in, memory)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), target.reshape(-1))
        return logits, loss


class EfficientLM(TiedLM):
    def __init__(self, vocab_size=VOCAB_SIZE, n_embd=1024, n_layer=10, n_head=16, n_kv_head=4, block_size=BLOCK):
        super().__init__(vocab_size, n_embd, block_size, use_positions=False)
        self.blocks = nn.ModuleList(
            EfficientBlock(n_embd, n_head, n_kv_head, block_size) for _ in range(n_layer)
        )
        self.norm = RMSNorm(n_embd)
        self.apply(_init_linear)

    def forward(self, idx, labels=None, pad_mask=None):
        if idx.size(1) > self.block_size:
            raise ValueError("sequence is longer than the context")
        x = self.run_blocks(self.blocks, self.token_plus_position(idx), pad_mask)
        logits = self.logits_from(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)), labels[:, 1:].reshape(-1))
        return logits, loss


def build_model(name, vocab_size=VOCAB_SIZE):
    if name == "decoder":
        model = DecoderLM(vocab_size=vocab_size)
    elif name == "encdec":
        model = EncoderDecoderLM(vocab_size=vocab_size)
    elif name == "hybrid":
        model = DecoderLM(vocab_size=vocab_size, windowed=True)
    elif name == "efficient":
        model = EfficientLM(vocab_size=vocab_size)
    else:
        raise ValueError("architecture must be decoder, encdec, hybrid, or efficient")
    total = count_parameters(model)
    if not 50_000_000 <= total <= 300_000_000:
        raise RuntimeError(f"{name} has {total} parameters, outside 50M-300M")
    return model


def sample_top_p(logits, top_p):
    sorted_logits, sorted_index = torch.sort(logits, descending=True, dim=-1)
    probs = torch.softmax(sorted_logits, dim=-1)
    cumulative = torch.cumsum(probs, dim=-1)
    remove = cumulative > top_p
    remove[..., 1:] = remove[..., :-1].clone()
    remove[..., 0] = False
    sorted_logits = sorted_logits.masked_fill(remove, float("-inf"))
    choice = torch.multinomial(torch.softmax(sorted_logits, dim=-1), num_samples=1)
    return torch.gather(sorted_index, -1, choice).squeeze(-1)


def _apply_stop(stop_fn, produced, finished, step, check_every):
    if stop_fn is None or (step + 1) % check_every != 0:
        return finished
    for position in torch.where(~finished)[0].tolist():
        if stop_fn(produced[position]):
            finished[position] = True
    return finished


@torch.no_grad()
def generate_causal(model, idx, max_new_tokens, temperature, top_p, eos_id=None, pad_id=0, stop_fn=None, check_every=32, min_new_tokens=0):
    pad_mask = idx != pad_id
    lengths = pad_mask.long().sum(dim=1).clamp(min=1)
    finished = torch.zeros(idx.size(0), dtype=torch.bool, device=idx.device)
    produced = [[] for _ in range(idx.size(0))]
    if idx.size(1) > model.block_size:
        overflow = idx.size(1) - model.block_size
        idx = idx[:, overflow:]
        pad_mask = pad_mask[:, overflow:]
        lengths = (lengths - overflow).clamp(min=1, max=model.block_size)
    for step in range(max_new_tokens):
        active = ~finished
        if not bool(active.any()):
            break
        if bool((active & (lengths >= model.block_size)).any()):
            idx = idx[:, 1:]
            pad_mask = pad_mask[:, 1:]
            lengths = (lengths - 1).clamp(min=1)
        if bool((lengths >= idx.size(1)).any()):
            idx = torch.cat([idx, idx.new_full((idx.size(0), 1), pad_id)], dim=1)
            pad_mask = torch.cat(
                [pad_mask, torch.zeros(idx.size(0), 1, dtype=torch.bool, device=idx.device)],
                dim=1,
            )
        limit = int(lengths.max().item())
        logits, _ = model(idx[:, :limit], pad_mask=pad_mask[:, :limit])
        last = (lengths - 1).clamp(min=0)
        row = torch.arange(idx.size(0), device=idx.device)
        next_logits = logits[row, last].float()
        if min_new_tokens and eos_id is not None:
            for position in range(idx.size(0)):
                if len(produced[position]) < min_new_tokens:
                    next_logits[position, eos_id] = -1e9
        if temperature <= 0:
            next_id = torch.argmax(next_logits, dim=-1)
        else:
            next_id = sample_top_p(next_logits / temperature, top_p)
        idx[row[active], lengths[active]] = next_id[active]
        pad_mask[row[active], lengths[active]] = True
        for position in torch.where(active)[0].tolist():
            produced[position].append(int(next_id[position]))
        lengths = lengths + active.long()
        if eos_id is not None:
            finished = finished | (active & (next_id == eos_id))
        finished = _apply_stop(stop_fn, produced, finished, step, check_every)
    return produced


@torch.no_grad()
def generate_encdec(model, idx, max_new_tokens, temperature, top_p, eos_id=None, pad_id=0, stop_fn=None, check_every=32):
    pad_mask = idx != pad_id
    lengths = pad_mask.long().sum(dim=1).clamp(min=1)
    if idx.size(1) > model.block_size:
        overflow = idx.size(1) - model.block_size
        idx = idx[:, overflow:]
        pad_mask = pad_mask[:, overflow:]
        lengths = (lengths - overflow).clamp(min=1)
    memory = model.encode(idx, pad_mask)
    row = torch.arange(idx.size(0), device=idx.device)
    dec = idx[row, lengths - 1].unsqueeze(1)
    produced = [[] for _ in range(idx.size(0))]
    finished = torch.zeros(idx.size(0), dtype=torch.bool, device=idx.device)
    for step in range(max_new_tokens):
        active = ~finished
        if not bool(active.any()):
            break
        if dec.size(1) >= model.block_size:
            dec = dec[:, 1:]
        logits = model.decode(dec, memory, memory_mask=pad_mask)
        next_logits = logits[:, -1, :]
        if temperature <= 0:
            next_id = torch.argmax(next_logits, dim=-1)
        else:
            next_id = sample_top_p(next_logits / temperature, top_p)
        next_id = torch.where(active, next_id, torch.full_like(next_id, eos_id or pad_id))
        dec = torch.cat([dec, next_id.unsqueeze(1)], dim=1)
        for position in torch.where(active)[0].tolist():
            produced[position].append(int(next_id[position]))
        if eos_id is not None:
            finished = finished | (active & (next_id == eos_id))
        finished = _apply_stop(stop_fn, produced, finished, step, check_every)
    return produced


def generate(model, idx, max_new_tokens, temperature, top_p, eos_id=None, pad_id=0, stop_fn=None, min_new_tokens=0):
    if isinstance(model, EncoderDecoderLM):
        return generate_encdec(model, idx, max_new_tokens, temperature, top_p, eos_id, pad_id, stop_fn)
    return generate_causal(model, idx, max_new_tokens, temperature, top_p, eos_id, pad_id, stop_fn,
                           min_new_tokens=min_new_tokens)


def _check():
    torch.manual_seed(0)
    for name in ("decoder", "encdec", "hybrid", "efficient"):
        model = build_model(name, vocab_size=VOCAB_SIZE)
        total = count_parameters(model)
        print(name, f"{total / 1e6:.2f}M")
        model.train()
        idx = torch.randint(1, 100, (2, 32))
        _logits, loss = model(idx, labels=idx)
        loss.backward()
        model.eval()
        produced = generate(model, idx[:, :8], max_new_tokens=3, temperature=0.0, top_p=1.0, eos_id=None, pad_id=0)
        assert len(produced) == 2
        assert all(len(row) == 3 for row in produced)
    print("phase1 model check passed")


In [ ]:
_register('phase1_models')


## 7. Instantiate the selected model


In [ ]:
BUILD = False
if BUILD:
    model = build_model(ARCH)
    print(ARCH, f"{count_parameters(model)/1e6:.2f}M")
else:
    print("BUILD=False (viva). Sizes:")
    for name, size in [("decoder", 110.18), ("hybrid", 110.18), ("efficient", 118.49), ("encdec", 106.26)]:
        print(f"  {name:10s} {size:.2f}M")


## 8. Training (Kaggle T4)

Next-token CE, AdamW, 50/25/15/10 SC/HC/tribunals/acts. Do not run in the viva.


In [ ]:
print("RUN_TRAINING", RUN_TRAINING)
print("Token mix", TOKEN_MIX)
if RUN_TRAINING:
    raise SystemExit("Train on a Kaggle T4, not on the viva laptop.")


## 9. Generation and zip

Greedy, stop at EOS, clip to 128 words. `prediction.json`: `{id: {text: continuation}}`.


In [ ]:
print("temperature", TEMPERATURE, "(0 = argmax)")
print("stop at EOS, then at most", TARGET_GENERATION_WORDS, "words")
